# 01 — Quickstart

Runs the DMET pipeline on LiH end to end and checks the numbers against
validated references.

LiH takes seconds. Start here before anything larger.

**Prerequisite:** `pip install -e ".[qiskit]"` (or `--no-deps` if you have
your own PySCF build). The GQE solver is not needed for this notebook.

### Orientation — what this run actually does to your disk

Everything lands under `project_dir`. After a full run you will have:

```
lih_run/
  results/
    step0_classical.pkl        classical reference energies
    step1_asf.pkl              active-space selection
    step2_hamiltonian.pkl      the embedding Hamiltonian  <- the important one
    step3_results.pkl          quantum solver output
    results_summary.csv        the human-readable table
    gqe_train.log              GQE training log, if the solver ran
    plots/                     figures from step 4
  cif_files/
```

`step2_hamiltonian.pkl` is the interface to everything downstream — the GQE
trainer, the determinant-selection tools in notebook 05, and
`tools/compare_pickles.py` all read it. If you keep one file from a run, keep
that one.

**Steps are cached.** Re-running a step that already has a pickle reuses it.
`force=True` (or `--force` on the CLI) recomputes and overwrites. During
development this is the single most common source of confusion: you change a
setting, re-run, and get the previous answer back because the cache was warm.

**Runtime.** LiH: seconds. N₂: seconds. ScH: minutes for steps 0–2, and the
GQE solver is a separate cost entirely (see notebook 03).

## 1. Verify the install first

`quenais-selftest` runs this whole pipeline and compares each quantity
against known-good values. If it fails, stop here — later numbers will not
mean anything.

If `quenais-selftest` fails, do not continue — nothing after it is meaningful.
The usual causes, in order of likelihood:

| symptom | cause |
|---|---|
| `ImportError: pyscf` | environment not active, or install was `--no-deps` without a PySCF build |
| energies off by ~1e-6 or worse | a different PySCF/libcint build; check `docs/limitations.md` before assuming a bug |
| `n_bath` mismatch on N₂ | the `adaptive_bath` fix is missing — you are on a pre-0.3 checkout |
| passes but very slow | thread count; set `OMP_NUM_THREADS` |

The selftest only runs `SELFTEST_SYSTEMS` (LiH by default) — it is a smoke
test, not full regression. For the full suite: `pytest tests/regression/`.

In [ ]:
!quenais-selftest

## 2. Build a Config

Settings are grouped rather than flat. Molecule identity, paths and the
solver choice live on `Config`; everything else sits in a settings group.

In [ ]:
from quenais import Config
from quenais.settings import AsfSettings, DmetSettings

cfg = Config(
    molecule="LiH",
    basis="sto-3g",
    project_dir="./lih_run",
    classical_methods=["HF", "MP2", "CCSD", "CCSD_T"],
    dmet=DmetSettings(reference="casci"),
)
cfg.validate().make_dirs().load_geometry()

print(cfg)
print("geometry :", cfg.geometry)
print("bath tol :", cfg.dmet.bath_tolerance)
print("max embed:", cfg.dmet.max_embed_orbs)

## 3. Step 0 — classical references

These are the answer key. Every bug found during development was caught by
a disagreement with a number produced here.

Note the `reproducibility` column: CASSCF and NEVPT2 are optimiser-dependent
and will not reproduce to tight tolerance across machines. See
`docs/limitations.md`.

**Which of these you can actually trust**, from
`tests/regression/reference_values.py`:

| tier | methods | reproducibility across machines |
|---|---|---|
| `DETERMINISTIC` | HF, MP2, CCSD, CCSD(T), DMET+CASCI, `ecore`, `mu`, σ spectrum | ~1e-10 — assert tightly |
| `OPTIMIZER_DEPENDENT` | CASSCF, NEVPT2 | ~1 mHa — assert loosely or not at all |
| `STOCHASTIC` | DMET+GQE | varies by design — only against a pinned seed |

This is not pedantry. Two CASSCF runs on *identical input* gave ScH
−752.680677 and −752.681604 (0.93 mHa apart), and the NEVPT2 built on top moved
3.6 mHa with it. Both are valid solutions to a non-convex optimisation. If you
write a regression test asserting CASSCF to 1e-6, it will fail on someone
else's machine and you will spend a day chasing a bug that is not there.

In [ ]:
from quenais.classical import runner

step0 = runner.main(cfg, force=True)

## 4. Step 1 — active space

ASF selects orbitals by entanglement entropy, then a degeneracy-aware gap
cutoff narrows the list. The cutoff is extended rather than allowed to
split a degenerate pair — splitting one breaks the molecule's symmetry.

In [ ]:
from quenais.active_space import finder

step1 = finder.main(cfg, force=True)
print()
print("active space :", f"({step1['nel']}e, {step1['n_active_orbs']}o)")
print("MOs          :", step1["mo_list"])

## 5. Step 2 — the DMET embedding

Watch two diagnostics in the output:

- **the Schmidt singular values.** These decide the bath. If none clears
  `bath_tolerance`, the correct answer is zero bath orbitals — not a
  fabricated bath from numerical noise.
- **the embedded electron count.** It comes from the reference density,
  not the active-space count. LiH's active space holds 2 electrons but its
  embedding space holds 4; using the active-space count roughly doubles
  the energy.

In [ ]:
from quenais.embedding import hamiltonian

step2 = hamiltonian.main(cfg, force=True)

## 6. The check that matters

`embedded_scf_check` runs a real, converged SCF on the embedding
Hamiltonian and compares it with the full-molecule UHF energy.

This is the single most diagnostic quantity in the pipeline. The `ecore`
self-consistency identity printed by some DMET codes is tautological —
`ecore` is *defined* as that difference, so it can never fail. This one
can.

Tolerance is `EMBEDDED_SCF_VS_UHF_TOL = 2e-7` Ha (validated at 1.3e-7 on ScH).

**If this check fails, stop.** It has caught, on its own, three separate real
bugs: an 8-orbital impurity in a 10-orbital basis leaving no environment
(0.73 Ha off, and the resulting curve was still smooth and monotonic); the
electron count taken from the active space instead of the reference density
(energy roughly doubled); and a mean-field density scaled by filling, leaving
fractional occupation (~1e-7 in `ecore`).

None of those raised an exception or failed to converge. This check is the only
thing that saw them.

In [ ]:
check = step2["embedded_scf_check"]
for k, v in check.items():
    print(f"  {k:15} {v}")

## 7. Compare against the validated value

LiH's DMET+CASCI reference is **−7.881246152 Ha**.

In [ ]:
from quenais.visualization.plots import true_embedding_casci

e_casci = true_embedding_casci(cfg)
reference = -7.881246152

print(f"DMET+CASCI : {e_casci:.9f} Ha")
print(f"reference  : {reference:.9f} Ha")
print(f"difference : {abs(e_casci - reference) * 1e3:.4f} mHa")

## 8. Step 4 — figures and summary

Produces whatever the available data supports. With no GQE log the GQE
figures are skipped rather than drawn empty.

In [ ]:
from quenais.visualization import plots

result = plots.main(cfg)
print()
print(open(result["results_summary"]).read())

## Same thing from the command line

```bash
quenais-run --molecule LiH --basis sto-3g --steps 0 1 2 4 --project-dir ./lih_run
```

---

## Troubleshooting — what actually goes wrong

**`n_bath = 0` and you did not expect it.**
Look at the singular values before assuming a bug. For N₂/STO-3G with the
golden active space, zero is *correct* — see notebook 02. Fabricating a bath
from sub-threshold values is a ~20 Ha error that produces no convergence
failure of any kind.

**The energy is roughly double what it should be.**
Electron count came from the active-space size rather than the reference
density trace. Check `n_alpha`/`n_beta` in the step-2 pickle against the
`ref_occ_*` sums — notebook 02 §"The electron count comes from the density"
prints exactly this comparison.

**Numbers changed between two identical runs.**
Expected for anything in the `STOCHASTIC` tier. *Not* expected for
`DETERMINISTIC` quantities — if those move, read `docs/reproducibility.md` §3
and §4 (ARPACK's random start vector, and degenerate orbitals fixed only up to
a rotation). Diff the fingerprints, not the energies.

**The active space looks wrong on a transition metal.**
It probably is. ASF's entropy thresholds are calibrated on main-group systems
and under-select for 3d elements — on ScH the automatic choice kept 4 orbitals
but only 2 active electrons. Use `--force-active-space` / `force_active_space`.
Notebook 04 §"If the active space looks wrong" covers the diagnosis.

## Where to go next

| you want to | notebook |
|---|---|
| understand the Schmidt decomposition and the diagnostics | `02_dmet_internals` |
| run the quantum solver | `03_gqe_solver` |
| run your own molecule | `04_full_workflow` |
| know whether your system can even show a quantum advantage | `05_determinant_selection` |